# 🎬 Kaggle 3D Pixar Cartoon Generator (LTX-Video / Dual T4 GPU)
### 🚀 100% Free Cloud Execution on Kaggle (30GB VRAM) — Zero Laptop Load!

**⚠️ IMPORTANT KAGGLE SETTINGS (Right Sidebar):**
1. **Accelerator:** Select **GPU T4 x2** (or GPU P100)
2. **Internet:** Toggle **Internet on** (Required to download models)

## ⚡ Step 1: Install High-Speed Video Libraries & Fetch Project Assets

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate sentencepiece protobuf imageio[ffmpeg]
!apt-get update -qq && apt-get install -y -qq ffmpeg

# Clone repository with our 5 generated 3D Pixar scenes
!rm -rf videogen
!git clone https://github.com/yashkumarsahu789/videogen.git

import os
print("✅ Setup complete! Verified GPU:")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 🎥 Step 2: Load LTX-Video Model (Image-to-Video DiT)

In [ ]:
import torch
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import load_image, export_to_video
from pathlib import Path

print("🚀 Loading LTX-Video DiT Pipeline on GPU...")
pipe = LTXImageToVideoPipeline.from_pretrained(
    "Lightricks/LTX-Video",
    torch_dtype=torch.bfloat16
)
pipe.to("cuda")
print("✅ LTX-Video successfully loaded on GPU!")

## 🎞️ Step 3: Animate All 5 Pixar Scenes into Razor-Sharp 3D Motion

In [ ]:
assets_dir = Path("videogen/video_output/cartoon_piku_mango_prank/assets")
output_dir = Path("output_clips")
output_dir.mkdir(parents=True, exist_ok=True)

scenes = [
    (
        "scene1_mango_bait.jpg",
        "clip1.mp4",
        "Cute baby monkey chewing sweet ripe mango on bamboo branch while goofy spotted leopard and chubby brown bear sneak forward through jungle bushes drooling in 3D Pixar cartoon style, smooth motion, high detail"
    ),
    (
        "scene2_catapult_launch.jpg",
        "clip2.mp4",
        "Giant bamboo branch snapping upward like a spring catapult, launching the screaming leopard and bear flying into the bright blue sky with flailing paws while baby monkey laughs heartily, fast cartoon action"
    ),
    (
        "scene3_beehive_helmet.jpg",
        "clip3.mp4",
        "Panicked chubby brown bear running in funny circles with a giant dripping beehive stuck on his head like a helmet, swarm of angry cartoon bees chasing him with buzzing trails, slapstick comedy"
    ),
    (
        "scene4_banana_slide.jpg",
        "clip4.mp4",
        "Goofy spotted leopard sprinting, comically slipping on giant yellow banana peels with legs twisting in pretzel shapes, colliding head-on into the running beehive bear with dust cloud explosion"
    ),
    (
        "scene5_river_splash_win.jpg",
        "clip5.mp4",
        "Huge water splash in turquoise jungle river, dizzy leopard and bear floating with yellow rubber ducks circling their heads, while baby monkey Piku winks and gives enthusiastic double thumbs-up"
    )
]

clip_paths = []
for img_file, out_name, prompt in scenes:
    in_path = assets_dir / img_file
    out_path = output_dir / out_name
    print(f"\n🎬 Generating Scene: {img_file}...")
    
    image = load_image(str(in_path)).resize((768, 512))
    generator = torch.Generator(device="cuda").manual_seed(42)
    
    video_frames = pipe(
        image=image,
        prompt=prompt,
        negative_prompt="blurry, low quality, distorted, extra limbs, dark, glitch",
        width=768,
        height=512,
        num_frames=25,
        num_inference_steps=30,
        generator=generator
    ).frames[0]
    
    export_to_video(video_frames, str(out_path), fps=8)
    clip_paths.append(str(out_path))
    print(f"✓ Generated: {out_name}")

print("\n🎉 ALL 5 SCENES GENERATED IN RAZOR-SHARP 3D MOTION!")

## 🎶 Step 4: Stitch All 5 Scenes into Final Master Movie

In [ ]:
# Write concat manifest
with open("concat_list.txt", "w") as f:
    for cp in clip_paths:
        f.write(f"file '{os.path.abspath(cp)}'\n")

final_output = "piku_3d_cartoon_master.mp4"
!ffmpeg -y -f concat -safe 0 -i concat_list.txt -c:v libx264 -pix_fmt yuv420p {final_output}
print(f"✅ Final video ready: {final_output}")

## 📥 Step 5: Play & Download Final Video

In [ ]:
from IPython.display import HTML, FileLink
from base64 import b64encode

mp4 = open(final_output, 'rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

print("🎬 PREVIEW YOUR CARTOON VIDEO:")
display(HTML(f"""
<video width=768 height=512 controls autoplay loop style="border-radius: 16px; border: 2px solid #10B981;">
    <source src="{data_url}" type="video/mp4">
</video>
"""))

print("\n⬇️ CLICK BELOW TO DOWNLOAD:")
display(FileLink(final_output))